# Decile portfolios from CNN predictions

In this section, I construct 10 decile portfolios from the model's predicted probability of an up move (`pred_prob_up`).

The logic is:

1. Use `end_date` as the portfolio formation date.
2. Compute the realized 5-day forward return from `close_now` and `close_future`.
3. On each rebalance date, sort all stocks into 10 deciles based on `pred_prob_up`.
4. Compute the equal-weight return of each decile.
5. Annualize the mean return and Sharpe ratio.

This follows the general setup in Jiang (2023) and my supervisor's paper, where stocks are sorted into decile portfolios using model predictions, the portfolios are equal-weighted, and held for five trading days.

In [760]:
import pandas as pd
import numpy as np
import os

## 1. Load the CSV and check the required columns

The file contains one row per stock-image observation.  
The most important columns here are:

- `end_date`: the date on which the image ends and the portfolio is formed
- `pred_prob_up`: the model's predicted probability that the future return is positive
- `close_now`: the close price at portfolio formation
- `close_future`: the close price 5 trading days later

I first load the data and make sure the required columns are present.

In [761]:
folders_new = [
    "color_candlestick_ma_bb_rsi",
    "color_candlestick_vol_0_ma_1_bb_1_rsi_0",
    "color_macd",
    "color_ohlc_rsi_half_panel",
    "color_ohlc_vol_0_ma_0_bb_0_rsi_0",
    "color_ohlc_volume_ma",
    "color_rsi_14",
    "color_vol_0_ma_0_bb_0_rsi_1",
    "color_vol_0_ma_1_bb_1_rsi_0",
    "color_volume",
    "grayscale_ohlc",
    "grayscale_ohlc_volume_ma",
]

folders_old = [
    "test_color_candlestick_vol_0_ma_0_bb_0_rsi_0",
    "test_color_candlestick_vol_0_ma_1_bb_0_rsi_0",
    "test_color_candlestick_vol_1_ma_0_bb_0_rsi_0",
    "test_color_vol_0_ma_1_bb_0_rsi_0",
    "test_color_vol_1_ma_0_bb_0_rsi_0",
    "test_color_vol_1_ma_1_bb_0_rsi_0",
    "test_res_96_color_candlestick_vol_1_ma_1_bb_0_rsi_0",
]

In [762]:
folder_index = 6
HEAD_FOLDER = folders_old[folder_index] # Change only once and run the entire program 


In [763]:
csv_path = os.path.join(HEAD_FOLDER, "test_predictions.csv") # for det gamle brug "test_predictions". For nye filer bare brug "predictions"

df = pd.read_csv(csv_path)

required_cols = [
    "ticker", "end_date", "close_now", "close_future", "pred_prob_up"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Parse date
df["end_date"] = pd.to_datetime(df["end_date"])

# Keep only the columns we need here
df = df[["ticker", "end_date", "close_now", "close_future", "pred_prob_up"]].copy()

print(df.head())
print("\nNumber of rows:", len(df))
print("Number of unique end_date values:", df["end_date"].nunique())
print("Number of unique tickers:", df["ticker"].nunique())

   ticker   end_date  close_now  close_future  pred_prob_up
0   10078 2001-02-01    31.1250       25.8750      0.509631
1   10078 2001-02-08    25.8750       27.1875      0.580964
2   10078 2001-02-15    27.1875       20.8125      0.521891
3   10078 2001-02-23    20.8125       19.6250      0.554619
4   10078 2001-03-02    19.6250       17.4375      0.524706

Number of rows: 71262
Number of unique end_date values: 725
Number of unique tickers: 701


## 2. Compute the realized 5-day forward return

For each stock observation, I compute the realized holding-period return as

$$
R_{i,t \to t+5} = \frac{\text{close\_future}_{i,t}}{\text{close\_now}_{i,t}} - 1
$$

This is the return that each decile portfolio earns over the next 5 trading days.

In [764]:
# Basic cleaning
df = df.dropna(subset=["ticker", "end_date", "close_now", "close_future", "pred_prob_up"]).copy()
df = df[(df["close_now"] > 0) & (df["close_future"] > 0)].copy()

# Remove duplicate stock-date observations if any
df = df.sort_values(["end_date", "ticker"]).drop_duplicates(subset=["ticker", "end_date"])

# Realized 5-day forward return
df["forward_return_5d"] = df["close_future"] / df["close_now"] - 1

print(df[["ticker", "end_date", "close_now", "close_future", "pred_prob_up", "forward_return_5d"]].head())

     ticker   end_date  close_now  close_future  pred_prob_up  \
0     10078 2001-02-01    31.1250        25.875      0.509631   
146   10104 2001-02-01    30.0625        27.125      0.537129   
292   10107 2001-02-01    62.3750        62.250      0.519333   
437   10108 2001-02-01    49.5000        52.800      0.498148   
756   10145 2001-02-01    47.7400        48.780      0.484012   

     forward_return_5d  
0            -0.168675  
146          -0.097713  
292          -0.002004  
437           0.066667  
756           0.021785  


## 3. Select portfolio formation dates

The non-overlapping image sequences were generated separately for each stock. As a result,
image endpoints are not perfectly synchronized across stocks. The data therefore contain a
dominant set of formation dates with broad cross-sectional coverage, as well as off-cycle
dates with relatively few available stocks.

To construct sufficiently broad decile portfolios, I retain formation dates with at least
300 available stocks. In the current sample, the retained dates contain at least 333 stocks
and follow the dominant non-overlapping five-trading-day sequence.

The retained stocks on each formation date are subsequently used to construct the decile
portfolios.

In [765]:
# Number of available stocks on each formation date
counts_per_date = (
    df.groupby("end_date")["ticker"]
      .nunique()
      .sort_index()
)

# Require sufficiently broad cross-sectional coverage.
# With 10 deciles, 300 stocks corresponds to at least about 30 stocks per decile.
MIN_STOCKS = 300

rebalance_dates = counts_per_date[
    counts_per_date >= MIN_STOCKS
].index.sort_values()

# Keep only observations belonging to the retained formation dates
weekly_df = df[
    df["end_date"].isin(rebalance_dates)
].copy()

# Summary of the retained portfolio sample
stocks_per_date = (
    weekly_df.groupby("end_date")["ticker"]
             .nunique()
)

print("Total formation dates in full sample:", df["end_date"].nunique())
print("Rebalance dates retained:", len(rebalance_dates))
print("Rows retained:", len(weekly_df))

print("\nStocks per retained formation date:")
print(stocks_per_date.describe())

Total formation dates in full sample: 725
Rebalance dates retained: 146
Rows retained: 54211

Stocks per retained formation date:
count    146.000000
mean     371.308219
std       35.402818
min      333.000000
25%      341.000000
50%      356.500000
75%      388.750000
max      464.000000
Name: ticker, dtype: float64


Since you want to check the retained formation dates after the cleaning/filtering, insert the check directly after the Section 3 code cell that creates rebalance_dates and weekly_df, and before Section 4 where you assign deciles.

One important detail: you want to check 5 trading periods apart, not simply (date2 - date1).days == 5, because weekends and holidays make calendar-day differences vary.

In [766]:
# Check that consecutive retained formation dates are 5 trading-date positions apart

all_dates = pd.DatetimeIndex(
    sorted(pd.to_datetime(df["end_date"].unique()))
)

date_position = pd.Series(
    np.arange(len(all_dates)),
    index=all_dates
)

rebalance_positions = date_position.loc[rebalance_dates]
gaps = rebalance_positions.diff().dropna()

print("Gap distribution between consecutive rebalance dates:")
print(gaps.value_counts().sort_index())

if (gaps == 5).all():
    print("\nAll consecutive rebalance dates are 5 trading-date positions apart.")
else:
    print("\nDates not 5 trading-date positions apart:")
    print(gaps[gaps != 5])

Gap distribution between consecutive rebalance dates:
4.0      1
5.0    144
Name: count, dtype: int64

Dates not 5 trading-date positions apart:
end_date
2001-02-08    4.0
dtype: float64


owever, based on the diagnostics we already ran, this particular check will report one gap = 4, for:

2001-01-30 -> 2001-02-06

even though we established that this period is valid. That's because all_dates contains only dates appearing somewhere as end_date, rather than a complete U.S. trading calendar.

So for your final notebook, I actually recommend a better check, which tests the thing you truly care about: whether the end price of one five-day period equals the starting price of the next period for stocks present in both periods.

Use this instead:

In [767]:
# Verify that consecutive retained formation dates represent
# consecutive non-overlapping 5-day holding periods

checks = []

for i in range(len(rebalance_dates) - 1):
    current_date = rebalance_dates[i]
    next_date = rebalance_dates[i + 1]

    current_period = (
        df[df["end_date"] == current_date]
        [["ticker", "close_future"]]
    )

    next_period = (
        df[df["end_date"] == next_date]
        [["ticker", "close_now"]]
    )

    comparison = current_period.merge(
        next_period,
        on="ticker",
        how="inner"
    )

    prices_match = np.isclose(
        comparison["close_future"],
        comparison["close_now"],
        rtol=1e-10,
        atol=1e-10
    )

    checks.append({
        "current_date": current_date,
        "next_date": next_date,
        "stocks_compared": len(comparison),
        "all_prices_match": prices_match.all()
    })

period_check = pd.DataFrame(checks)

print("Number of transitions checked:", len(period_check))
print("All periods consecutive:", period_check["all_prices_match"].all())

if not period_check["all_prices_match"].all():
    print("\nProblematic transitions:")
    print(period_check[~period_check["all_prices_match"]])

Number of transitions checked: 145
All periods consecutive: True


In [768]:
r = weekly_df["forward_return_5d"].dropna()

print(f"Fraction positive: {(r > 0).mean():.4%}")
print(f"Average positive return: {r[r > 0].mean():.4%}")
print(f"Average negative return: {r[r < 0].mean():.4%}")
print(f"Overall average return: {r.mean():.4%}")

Fraction positive: 50.7886%
Average positive return: 4.0260%
Average negative return: -4.3459%
Overall average return: -0.0770%


## 4. Assign stocks to 10 deciles on each rebalance date

On each `end_date`, I sort stocks by `pred_prob_up`:

- Decile 1 = lowest predicted probability of going up
- Decile 10 = highest predicted probability of going up

I use `pd.qcut()` to split the cross-section into 10 approximately equal-sized groups.

A small practical issue is that some stocks can have identical prediction values.  
To make `qcut()` stable, I first rank the predictions using `rank(method="first")`.

In [769]:
# Assign stocks to deciles separately on each formation date
weekly_df = weekly_df.copy()

weekly_df["decile"] = (
    weekly_df
    .groupby("end_date")["pred_prob_up"]
    .transform(
        lambda s: pd.qcut(
            s.rank(method="first"),
            q=10,
            labels=False
        ) + 1
    )
    .astype(int)
)

print(weekly_df.head(10))
print("\nNumber of rows:", len(weekly_df))

      ticker   end_date  close_now  close_future  pred_prob_up  \
0      10078 2001-02-01   31.12500       25.8750      0.509631   
146    10104 2001-02-01   30.06250       27.1250      0.537129   
292    10107 2001-02-01   62.37500       62.2500      0.519333   
437    10108 2001-02-01   49.50000       52.8000      0.498148   
756    10145 2001-02-01   47.74000       48.7800      0.484012   
902    10147 2001-02-01   77.60000       59.5000      0.525959   
1191   10299 2001-02-01   61.23438       56.6875      0.550178   
1337   10324 2001-02-01   85.06250       88.0625      0.552254   
1491   10401 2001-02-01   24.46000       22.9000      0.514828   
1637   10516 2001-02-01   14.74000       15.9600      0.457010   

      forward_return_5d  decile  
0             -0.168675       6  
146           -0.097713       8  
292           -0.002004       7  
437            0.066667       4  
756            0.021785       3  
902           -0.233247       7  
1191          -0.074254       9  
1

In [770]:
print("\nColumns after assigning deciles:")
print(weekly_df.columns.tolist())


Columns after assigning deciles:
['ticker', 'end_date', 'close_now', 'close_future', 'pred_prob_up', 'forward_return_5d', 'decile']


## 5. Compute equal-weight decile returns on each rebalance date

For each formation date and decile, I take the simple average of the realized 5-day forward returns across all stocks in that decile.

This gives me one 5-day portfolio return for each decile on each rebalance date.

In [771]:
decile_returns_by_date = (
    weekly_df
    .groupby(["end_date", "decile"])["forward_return_5d"]
    .mean()
    .reset_index()
    .sort_values(["end_date", "decile"])
)

print(decile_returns_by_date.head(15))

     end_date  decile  forward_return_5d
0  2001-02-01       1          -0.021118
1  2001-02-01       2          -0.022854
2  2001-02-01       3          -0.023949
3  2001-02-01       4          -0.024177
4  2001-02-01       5          -0.019536
5  2001-02-01       6          -0.039479
6  2001-02-01       7          -0.028335
7  2001-02-01       8          -0.017084
8  2001-02-01       9          -0.016591
9  2001-02-01      10          -0.029643
10 2001-02-08       1          -0.008595
11 2001-02-08       2          -0.004978
12 2001-02-08       3          -0.010675
13 2001-02-08       4          -0.007612
14 2001-02-08       5           0.001656


## 6. Put the decile returns into wide format and construct High-minus-Low

Now I reshape the data so that each column is one decile:

- column 1 = Low
- column 10 = High

Then I create the spread portfolio:

$$H-L = \text{Decile 10} - \text{Decile 1}$$

This is the same long-short spread that is reported in the papers.

In [772]:
decile_matrix = decile_returns_by_date.pivot(
    index="end_date",
    columns="decile",
    values="forward_return_5d"
).sort_index()

# Add High-minus-Low spread
decile_matrix["H-L"] = decile_matrix[10] - decile_matrix[1]

# Optional: rename columns for nicer display
decile_matrix = decile_matrix.rename(columns={1: "Low", 10: "High"})

print(decile_matrix.head())

decile           Low         2         3         4         5         6  \
end_date                                                                 
2001-02-01 -0.021118 -0.022854 -0.023949 -0.024177 -0.019536 -0.039479   
2001-02-08 -0.008595 -0.004978 -0.010675 -0.007612  0.001656  0.009297   
2001-02-15 -0.035706 -0.055009 -0.043587 -0.068542 -0.060932 -0.058045   
2001-02-23  0.002134  0.005424 -0.019493 -0.017033 -0.021633 -0.007100   
2001-03-02  0.008517 -0.000806  0.007918  0.011370  0.001026  0.012784   

decile             7         8         9      High       H-L  
end_date                                                      
2001-02-01 -0.028335 -0.017084 -0.016591 -0.029643 -0.008525  
2001-02-08  0.002658 -0.009203  0.006974  0.010621  0.019216  
2001-02-15 -0.073397 -0.082521 -0.071286 -0.042024 -0.006318  
2001-02-23 -0.012431 -0.012750 -0.006242  0.011317  0.009182  
2001-03-02 -0.015876 -0.014861 -0.001643 -0.006629 -0.015147  


## 7. Compute annualized return and annualized Sharpe ratio

Each row in `decile_matrix` is a **5-trading-day portfolio return**.

So I annualize using:

$$
\text{periods per year} = \frac{252}{5}
$$

Then:

$$
\text{Annualized Return} = \bar{r}_{5d} \times \frac{252}{5}
$$

$$
\text{Annualized Sharpe} = \frac{\bar{r}_{5d}}{\sigma_{5d}} \times \sqrt{\frac{252}{5}}
$$

This is the standard way to annualize fixed-horizon portfolio returns.

In [773]:
periods_per_year = 252 / 5  # 5-trading-day holding period

def annualized_stats(return_series):
    s = pd.Series(return_series).dropna()
    
    mean_5d = s.mean()
    std_5d = s.std(ddof=1)
    
    ann_return = mean_5d * periods_per_year
    
    if std_5d == 0 or np.isnan(std_5d):
        ann_sharpe = np.nan
    else:
        ann_sharpe = (mean_5d / std_5d) * np.sqrt(periods_per_year)
    
    return pd.Series({
        "Mean_5d_Return": mean_5d,
        "Std_5d_Return": std_5d,
        "Annualized_Return": ann_return,
        "Annualized_Sharpe": ann_sharpe,
        "N_periods": len(s)
    })

summary = decile_matrix.apply(annualized_stats, axis=0).T
summary

,Mean_5d_Return,Std_5d_Return,Annualized_Return,Annualized_Sharpe,N_periods
decile,,,,,
Low,-0.003806,0.029374,-0.191830,-0.919897,146.0
2,-0.003405,0.031836,-0.171625,-0.759362,146.0
3,-0.002585,0.033255,-0.130299,-0.551903,146.0
4,-0.000706,0.031673,-0.035584,-0.158253,146.0
5,-0.000201,0.034192,-0.010130,-0.041734,146.0
6,0.000552,0.034573,0.027833,0.113397,146.0
7,-0.001729,0.035312,-0.087149,-0.347632,146.0
8,0.000079,0.035893,0.003999,0.015692,146.0
9,0.003282,0.036061,0.165400,0.646076,146.0


## 8. Make the final table look like the paper

To make the output easier to compare with the paper tables, I keep only the annualized return and annualized Sharpe ratio, and I convert the return to percent.

In [774]:
final_table = summary[["Annualized_Return", "Annualized_Sharpe", "N_periods"]].copy()
final_table["Annualized_Return_pct"] = final_table["Annualized_Return"] * 100

# Put columns in a nicer order
final_table = final_table[["Annualized_Return_pct", "Annualized_Sharpe", "N_periods"]]

# Optional: reorder rows
desired_order = ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"]
final_table = final_table.reindex([x for x in desired_order if x in final_table.index])

print(final_table.round(4))

        Annualized_Return_pct  Annualized_Sharpe  N_periods
decile                                                     
Low                  -19.1830            -0.9199      146.0
2                    -17.1625            -0.7594      146.0
3                    -13.0299            -0.5519      146.0
4                     -3.5584            -0.1583      146.0
5                     -1.0130            -0.0417      146.0
6                      2.7833             0.1134      146.0
7                     -8.7149            -0.3476      146.0
8                      0.3999             0.0157      146.0
9                     16.5400             0.6461      146.0
High                  24.1913             0.9047      146.0
H-L                   43.3743             2.6476      146.0


## 9. Interpretation of the output

The final table should be read as follows:

- `Low` is the decile with the lowest predicted probability of an up move.
- `High` is the decile with the highest predicted probability of an up move.
- `H-L` is a long-short strategy that buys the highest decile and shorts the lowest decile.
- `Annualized_Return_pct` is the annualized mean return in percent.
- `Annualized_Sharpe` is the annualized Sharpe ratio.

If the model is useful, I should generally see returns and Sharpe ratios improve as I move from `Low` to `High`.

In [775]:
# Average number of stocks in each decile
avg_names_per_decile = (
    weekly_df.groupby(["end_date", "decile"]).size()
    .groupby("decile")
    .mean()
)

print("Average number of stocks per decile:")
print(avg_names_per_decile.round(2))

# Check monotonicity visually
display_cols = [c for c in ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"] if c in decile_matrix.columns]
display(decile_matrix[display_cols].head())

Average number of stocks per decile:
decile
1     37.60
2     37.05
3     37.01
4     37.03
5     37.21
6     36.89
7     36.99
8     37.05
9     37.01
10    37.47
dtype: float64


decile,Low,2,3,4,5,6,7,8,9,High,H-L
end_date,,,,,,,,,,,
2001-02-01,-0.021118,-0.022854,-0.023949,-0.024177,-0.019536,-0.039479,-0.028335,-0.017084,-0.016591,-0.029643,-0.008525
2001-02-08,-0.008595,-0.004978,-0.010675,-0.007612,0.001656,0.009297,0.002658,-0.009203,0.006974,0.010621,0.019216
2001-02-15,-0.035706,-0.055009,-0.043587,-0.068542,-0.060932,-0.058045,-0.073397,-0.082521,-0.071286,-0.042024,-0.006318
2001-02-23,0.002134,0.005424,-0.019493,-0.017033,-0.021633,-0.007100,-0.012431,-0.012750,-0.006242,0.011317,0.009182
2001-03-02,0.008517,-0.000806,0.007918,0.011370,0.001026,0.012784,-0.015876,-0.014861,-0.001643,-0.006629,-0.015147


## Always-long equal-weight benchmark

As a reference benchmark, I also calculate the performance of an **always-long, equal-weight portfolio across all stocks**.

This benchmark does **not** sort stocks into deciles.  
Instead, on each portfolio formation date, it simply buys **all available stocks** and assigns each stock the same weight.

That is why this benchmark produces only **one portfolio return series**, and therefore only **one annualized return** and **one Sharpe ratio**.

### Step 1: Compute the 5-day forward return for each stock

For each stock $i$ on formation date $t$, the realized 5-day forward return is:

$$
r_{i,t} = \frac{\text{close\_future}_{i,t}}{\text{close\_now}_{i,t}} - 1
$$

where:

- $\text{close\_now}_{i,t}$ is the closing price at the portfolio formation date
- $\text{close\_future}_{i,t}$ is the closing price 5 trading days later

### Step 2: Compute the equal-weight benchmark return on each date

On each formation date $t$, the always-long benchmark return is the simple average of all stock returns on that date:

$$
r^{EW}_t = \frac{1}{N_t} \sum_{i=1}^{N_t} r_{i,t}
$$

where $N_t$ is the number of available stocks on date $t$.

So instead of creating 10 decile portfolios, I create only **one** portfolio each period:

- long all stocks
- equal weight each stock
- hold for 5 trading days

This gives a time series of benchmark returns:

$$
r^{EW}_{t_1}, r^{EW}_{t_2}, r^{EW}_{t_3}, \dots
$$

### Step 3: Annualize the mean return

Because each portfolio is held for 5 trading days, the number of holding periods per year is approximately:

$$
\frac{252}{5}
$$

where 252 is the standard number of trading days in a year.

The annualized return is therefore:

$$
\text{Annualized Return} = \bar{r}_{5d} \times \frac{252}{5}
$$

where $\bar{r}_{5d}$ is the average 5-day benchmark return across all periods.

### Step 4: Annualize the Sharpe ratio

The Sharpe ratio measures return relative to volatility.

Let $\sigma_{5d}$ denote the standard deviation of the 5-day benchmark returns.  
Then the annualized Sharpe ratio is:

$$
\text{Annualized Sharpe} = \frac{\bar{r}_{5d}}{\sigma_{5d}} \times \sqrt{\frac{252}{5}}
$$

### Why does this benchmark only give one return and one Sharpe ratio?

The decile analysis gives many values because stocks are split into many portfolios:

- Decile 1
- Decile 2
- ...
- Decile 10
- High-minus-Low

So each decile has its own return series and its own Sharpe ratio.

In contrast, the always-long benchmark does **not** split stocks into groups.  
It simply averages all stocks into one equal-weight portfolio on each date.

Therefore, it produces:

- one portfolio return series
- one annualized return
- one annualized Sharpe ratio

### Interpretation

This benchmark is useful because it shows how well a simple passive strategy performs without using the model.

If the model is useful in a long-only sense, then the **High decile** should ideally outperform this benchmark in terms of:

- annualized return
- Sharpe ratio

If the model is useful as a ranking model, then returns should generally improve from the **Low** decile to the **High** decile, and the **High-minus-Low** spread should be positive and economically meaningful.

In [776]:
# Always-long benchmark using the SAME sample as the decile portfolios

benchmark_df = weekly_df.copy()

benchmark_df["end_date"] = pd.to_datetime(benchmark_df["end_date"])
benchmark_df = benchmark_df.dropna(subset=["end_date", "forward_return_5d"])

ew_returns = (
    benchmark_df
    .groupby("end_date")["forward_return_5d"]
    .mean()
    .sort_index()
)

# Make sure it uses exactly the same dates as decile_matrix
ew_returns = ew_returns.reindex(decile_matrix.index).dropna()

periods_per_year = 252 / 5

annualized_return = ew_returns.mean() * periods_per_year
annualized_sharpe = (
    ew_returns.mean() / ew_returns.std(ddof=1)
) * np.sqrt(periods_per_year)

print("Always-long equal-weight benchmark, same sample as deciles")
print(f"Annualized return: {annualized_return:.4f}  ({annualized_return*100:.2f}%)")
print(f"Annualized Sharpe: {annualized_sharpe:.4f}")
print(f"Number of periods: {len(ew_returns)}")

Always-long equal-weight benchmark, same sample as deciles
Annualized return: -0.0187  (-1.87%)
Annualized Sharpe: -0.0812
Number of periods: 146


In [777]:
import pandas as pd
import numpy as np
from scipy import stats

# --------------------------------------------------
# t-test / p-value table for deciles
# Uses:
#   decile_matrix
# --------------------------------------------------

# Safe copy
test_df = decile_matrix.copy()

# Make sure dates are sorted
test_df.index = pd.to_datetime(test_df.index)
test_df = test_df.sort_index()

# Nice row order
row_order = [c for c in ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"] if c in test_df.columns]

rows = []
for col in row_order:
    s = pd.to_numeric(test_df[col], errors="coerce").dropna()

    # one-sample t-test: mean return = 0
    t_stat, p_val = stats.ttest_1samp(s, popmean=0.0, nan_policy="omit")

    rows.append({
        "Decile": str(col),
        "t": t_stat,
        "p": p_val
    })

t_table = pd.DataFrame(rows).set_index("Decile")
t_table = t_table.round(4)
t_table.columns = pd.MultiIndex.from_product([["All"], t_table.columns])

display(t_table)

All        
             t       p
Decile                
Low    -1.5657  0.1196
2      -1.2924  0.1983
3      -0.9393  0.3491
4      -0.2693  0.7880
5      -0.0710  0.9435
6       0.1930  0.8472
7      -0.5917  0.5550
8       0.0267  0.9787
9       1.0996  0.2733
High    1.5398  0.1258
H-L     4.5062  0.0000

In [778]:
final_table = summary[["Annualized_Return", "Annualized_Sharpe", "N_periods"]].copy()
final_table["Annualized_Return_pct"] = final_table["Annualized_Return"] * 100

# Put columns in a nicer order
final_table = final_table[["Annualized_Return_pct", "Annualized_Sharpe", "N_periods"]]

# Optional: reorder rows
desired_order = ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"]
final_table = final_table.reindex([x for x in desired_order if x in final_table.index])

print(final_table.round(4))
print(folders_old[folder_index])

        Annualized_Return_pct  Annualized_Sharpe  N_periods
decile                                                     
Low                  -19.1830            -0.9199      146.0
2                    -17.1625            -0.7594      146.0
3                    -13.0299            -0.5519      146.0
4                     -3.5584            -0.1583      146.0
5                     -1.0130            -0.0417      146.0
6                      2.7833             0.1134      146.0
7                     -8.7149            -0.3476      146.0
8                      0.3999             0.0157      146.0
9                     16.5400             0.6461      146.0
High                  24.1913             0.9047      146.0
H-L                   43.3743             2.6476      146.0
test_res_96_color_candlestick_vol_1_ma_1_bb_0_rsi_0
